<a href="https://colab.research.google.com/github/belhadjaissa07-droid/Chicken-disease-classification-project/blob/main/keras_tuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fashion-MNIST Classification with CNN and KerasTuner


In [1]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.6 MB/s eta 0:00:00


## 1. Imports of needed libraries



In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


In [3]:
fashion_mnist = keras.datasets.fashion_mnist

## 2. Load Fashion-MNIST Dataset

In [4]:
(train_img,train_labels),(test_img,test_labels) = fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## 3. Data Preprocessing


In [5]:
train_img = train_img / 255.0
test_img = test_img / 255.0

In [6]:
train_img =train_img.reshape(len(train_img),28,28,1)
test_img =test_img.reshape(len(test_img),28,28,1)

In [7]:
train_img.shape

(60000, 28, 28, 1)

## 4. CNN Model

In [8]:
def build_model(hp):

    model = keras.Sequential([

        keras.layers.Conv2D(
            filters=hp.Int(
                'conv1_filter',
                min_value=32,
                max_value=128,
                step=16
            ),
            kernel_size=hp.Choice(
                'conv_1_kernel',
                values=[3, 5]
            ),
            activation='relu',
            input_shape=(28, 28, 1)
        ),

        keras.layers.Conv2D(
            filters=hp.Int(
                'conv_2_filter',
                min_value=32,
                max_value=64,
                step=16
            ),
            kernel_size=hp.Choice(
                'conv_2_kernel',
                values=[3, 5]
            ),
            activation='relu'
        ),

        keras.layers.Flatten(),

        keras.layers.Dense(
            units=hp.Int(
                'dense_1_units',
                min_value=32,
                max_value=128,
                step=16
            ),
            activation='relu'
        ),

        keras.layers.Dense(
            10,
            activation='softmax'
        )
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice(
                'learning_rate',
                values=[1e-2, 1e-3]
            )
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [9]:
from kerastuner import RandomSearch
from kerastuner.engine.hyperparameters import HyperParameters

/tmp/ipykernel_569/556418634.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner import RandomSearch


In [10]:
tuner_search=RandomSearch(build_model,
                          objective='val_accuracy',
                          max_trials=5,directory='output',project_name="Mnist Fashion")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## 5. Hyperparameter Tuning

In [11]:
tuner_search.search(train_img,train_labels,epochs=3,validation_split=0.1)

Trial 5 Complete [00h 00m 35s]
val_accuracy: 0.8790000081062317

Best val_accuracy So Far: 0.9100000262260437
Total elapsed time: 00h 02m 48s


In [12]:
model=tuner_search.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 128)    │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 48)     │        55,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 23232)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 48)             │     1,115,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,174,346 (4.48 MB)

 Trainable params: 1,174,346 (4.48 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Training

In [14]:
history = model.fit(train_img,train_labels,epochs = 5,validation_split=0.1)

Epoch 1/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.9291 - loss: 0.1902 - val_accuracy: 0.9008 - val_loss: 0.2778
Epoch 2/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9433 - loss: 0.1534 - val_accuracy: 0.9137 - val_loss: 0.2709
Epoch 3/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9539 - loss: 0.1214 - val_accuracy: 0.9153 - val_loss: 0.3003
Epoch 4/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9642 - loss: 0.0963 - val_accuracy: 0.9088 - val_loss: 0.3169
Epoch 5/5
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9706 - loss: 0.0777 - val_accuracy: 0.9127 - val_loss: 0.3799


## 8. Evaluation

In [15]:
test_loss, test_accuracy = model.evaluate(
    test_img,
    test_labels
)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9049 - loss: 0.4232


In [16]:
print(test_loss)
print(test_accuracy)

0.423201322555542
0.9049000144004822


In [17]:
y_pred = model.predict(test_img)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [18]:
y_pred_classes = np.argmax(y_pred, axis=1)

## 9. Confusion Matrix


In [19]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    test_labels,
    y_pred_classes
)

In [20]:
print(cm)

[[882   0  12  30   4   1  60   0  11   0]
 [  1 970   1  21   2   0   3   0   2   0]
 [ 23   0 821  10  85   0  58   0   3   0]
 [ 13   0  10 940  25   0  10   0   2   0]
 [  2   0  55  29 859   0  51   0   4   0]
 [  0   0   0   1   0 967   0  26   0   6]
 [132   1  54  36  79   0 687   0  11   0]
 [  0   0   0   0   0   4   0 985   0  11]
 [  2   0   1   3   4   2   4   2 982   0]
 [  1   0   0   0   0   3   0  40   0 956]]


In [21]:
print(history.history['accuracy'])
print(history.history['val_accuracy'])

[0.9291296005249023, 0.9432963132858276, 0.9539074301719666, 0.9641851782798767, 0.9706110954284668]
[0.9008333086967468, 0.9136666655540466, 0.9153333306312561, 0.9088333249092102, 0.9126666784286499]
